# Alinta Energy case study — Victorian electricity demand prediction

**Assignment question:** Can historical electricity demand and weather conditions be used to predict Victorian electricity demand?

This notebook:
1. reads the AEMO 12-month Operational Demand ZIP you download manually;
2. filters the data to `VIC1`;
3. aggregates AEMO demand to hourly average MW;
4. downloads matching Melbourne historical weather from Open-Meteo;
5. aligns weather to fixed AEST/NEM market time;
6. creates calendar and lagged-demand features;
7. uses a chronological 80/20 train-test split;
8. compares a naive previous-day baseline, Decision Tree Regressor, and Random Forest Regressor;
9. reports MAE, RMSE, R² and high-demand (peak) MAE;
10. tests whether adding weather improves the Random Forest;
11. saves tables and figures for the assignment appendix.

**Fixed analysis period:** 1 September 2025 to 31 July 2026.

Do not invent any result. Copy the exact values produced by this notebook into Part 1.3.

## Before running

Download the newest ZIP file shown at:

`https://nemweb.com.au/Reports/Current/Operational_Demand/ACTUAL_5MIN/`

The filename begins with:

`PUBLIC_ACTUAL_OPERATIONAL_DEMAND_5MIN_`

Keep that ZIP file unchanged. Run the next cell and select the ZIP when Colab asks you to upload a file.

In [ ]:
# STEP 1 — Import packages and upload the AEMO ZIP
import io
import csv
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from google.colab import files
except ImportError:
    raise RuntimeError("Open this notebook in Google Colab so the upload/download cells work.")

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if len(zip_names) != 1:
    raise ValueError("Please upload exactly ONE AEMO .zip file.")

AEMO_ZIP = zip_names[0]
print("Uploaded:", AEMO_ZIP)

In [ ]:
# STEP 2 — Parse AEMO's CSV format robustly

def _decode_bytes(raw):
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", errors="replace")

def parse_aemo_operational_demand_zip(zip_path):
    records = []

    with zipfile.ZipFile(zip_path) as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            raise ValueError("No CSV file was found inside the AEMO ZIP.")

        for member in csv_members:
            text = _decode_bytes(zf.read(member))
            rows = list(csv.reader(io.StringIO(text)))
            member_records = []

            # AEMO MMS files commonly use I (header) and D (data) control rows.
            active_indexes = None
            for row in rows:
                if not row:
                    continue

                cleaned = [x.strip() for x in row]

                if (
                    cleaned[0] == "I"
                    and "INTERVAL_DATETIME" in cleaned
                    and "REGIONID" in cleaned
                    and "OPERATIONAL_DEMAND" in cleaned
                ):
                    active_indexes = {
                        "timestamp": cleaned.index("INTERVAL_DATETIME"),
                        "region": cleaned.index("REGIONID"),
                        "demand": cleaned.index("OPERATIONAL_DEMAND"),
                    }
                    continue

                if cleaned[0] == "D" and active_indexes is not None:
                    max_i = max(active_indexes.values())
                    if len(cleaned) <= max_i:
                        continue
                    member_records.append({
                        "timestamp": cleaned[active_indexes["timestamp"]],
                        "region": cleaned[active_indexes["region"]],
                        "demand_mw": cleaned[active_indexes["demand"]],
                    })

            # Fallback for a normal CSV with a conventional header.
            if not member_records:
                try:
                    normal = pd.read_csv(io.StringIO(text))
                    needed = {"INTERVAL_DATETIME", "REGIONID", "OPERATIONAL_DEMAND"}
                    if needed.issubset(normal.columns):
                        member_records = (
                            normal.rename(columns={
                                "INTERVAL_DATETIME": "timestamp",
                                "REGIONID": "region",
                                "OPERATIONAL_DEMAND": "demand_mw",
                            })[["timestamp", "region", "demand_mw"]]
                            .to_dict("records")
                        )
                except Exception:
                    pass

            records.extend(member_records)

    if not records:
        raise ValueError(
            "Could not find INTERVAL_DATETIME, REGIONID and OPERATIONAL_DEMAND "
            "inside the ZIP. Check that you downloaded the ACTUAL_5MIN Operational Demand file."
        )

    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["demand_mw"] = pd.to_numeric(df["demand_mw"], errors="coerce")
    df["region"] = df["region"].astype(str).str.strip()

    df = df.dropna(subset=["timestamp", "demand_mw"])
    return df

aemo_raw = parse_aemo_operational_demand_zip(AEMO_ZIP)

print("Parsed rows:", f"{len(aemo_raw):,}")
print("Regions:", sorted(aemo_raw["region"].dropna().unique())[:10])
print("Date range:", aemo_raw["timestamp"].min(), "to", aemo_raw["timestamp"].max())
aemo_raw.head()

In [ ]:
# STEP 3 — Filter VIC1 and use a fixed study period

START = pd.Timestamp("2025-09-01 00:00:00")
END_EXCLUSIVE = pd.Timestamp("2026-08-01 00:00:00")

vic_5min = (
    aemo_raw.loc[aemo_raw["region"].eq("VIC1"), ["timestamp", "demand_mw"]]
    .drop_duplicates(subset="timestamp", keep="last")
    .sort_values("timestamp")
)

if vic_5min.empty:
    raise ValueError("No VIC1 rows were found. Re-check the downloaded AEMO file.")

if vic_5min["timestamp"].min() > START or vic_5min["timestamp"].max() < (END_EXCLUSIVE - pd.Timedelta(minutes=5)):
    raise ValueError(
        "The uploaded rolling file does not fully cover 2025-09-01 to 2026-07-31. "
        "Download the newest AEMO ACTUAL_5MIN file and run again."
    )

vic_5min = vic_5min[
    (vic_5min["timestamp"] >= START) &
    (vic_5min["timestamp"] < END_EXCLUSIVE)
].copy()

demand_hourly = (
    vic_5min
    .set_index("timestamp")["demand_mw"]
    .resample("1h")
    .mean()
    .rename("demand_mw")
    .reset_index()
)

print("VIC1 five-minute rows in study period:", f"{len(vic_5min):,}")
print("Hourly demand rows before quality filtering:", f"{len(demand_hourly):,}")
print("Missing hourly demand values:", int(demand_hourly["demand_mw"].isna().sum()))
demand_hourly.head()

In [ ]:
# STEP 4 — Download matching Melbourne weather from Open-Meteo
#
# AEMO/NEM market timestamps use fixed AEST (UTC+10). We therefore ask Open-Meteo
# to label the weather timestamps in Australia/Brisbane time, which is also fixed UTC+10,
# so daylight-saving time cannot shift the merge by one hour.

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -37.8136,
    "longitude": 144.9631,
    "start_date": "2025-09-01",
    "end_date": "2026-07-31",
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
    ]),
    "timezone": "Australia/Brisbane",
}

response = requests.get(WEATHER_URL, params=params, timeout=60)
response.raise_for_status()
weather_json = response.json()

weather = pd.DataFrame(weather_json["hourly"])
weather = weather.rename(columns={"time": "timestamp"})
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
]
for col in weather_cols:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

print("Weather rows:", f"{len(weather):,}")
print("Weather date range:", weather["timestamp"].min(), "to", weather["timestamp"].max())
print("Missing values by weather field:")
display(weather[weather_cols].isna().sum().to_frame("missing"))
weather.head()

In [ ]:
# STEP 5 — Merge electricity demand and weather, then engineer features

merged = demand_hourly.merge(weather, on="timestamp", how="inner")
merged = merged.sort_values("timestamp").reset_index(drop=True)

merged["hour"] = merged["timestamp"].dt.hour
merged["day_of_week"] = merged["timestamp"].dt.dayofweek
merged["is_weekend"] = (merged["day_of_week"] >= 5).astype(int)

merged["hour_sin"] = np.sin(2 * np.pi * merged["hour"] / 24)
merged["hour_cos"] = np.cos(2 * np.pi * merged["hour"] / 24)
merged["dow_sin"] = np.sin(2 * np.pi * merged["day_of_week"] / 7)
merged["dow_cos"] = np.cos(2 * np.pi * merged["day_of_week"] / 7)

# Historical demand features use earlier timestamps only.
merged["demand_lag_1h"] = merged["demand_mw"].shift(1)
merged["demand_lag_24h"] = merged["demand_mw"].shift(24)
merged["demand_lag_168h"] = merged["demand_mw"].shift(168)
merged["demand_roll24h"] = merged["demand_mw"].shift(1).rolling(24).mean()

FEATURE_COLS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "demand_lag_1h",
    "demand_lag_24h",
    "demand_lag_168h",
    "demand_roll24h",
]

WEATHER_COLS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
]

model_df = merged.dropna(subset=FEATURE_COLS + ["demand_mw"]).copy()

print("Merged hourly rows:", f"{len(merged):,}")
print("ML-ready rows after lag/missing-value removal:", f"{len(model_df):,}")
print("Study range after lag creation:", model_df["timestamp"].min(), "to", model_df["timestamp"].max())
model_df.head()

In [ ]:
# STEP 6 — Chronological 80/20 split (NO random shuffling)

split_index = int(len(model_df) * 0.80)

train = model_df.iloc[:split_index].copy()
test = model_df.iloc[split_index:].copy()

X_train = train[FEATURE_COLS]
y_train = train["demand_mw"]

X_test = test[FEATURE_COLS]
y_test = test["demand_mw"]

print("Training rows:", f"{len(train):,}", "|", train["timestamp"].min(), "to", train["timestamp"].max())
print("Test rows:", f"{len(test):,}", "|", test["timestamp"].min(), "to", test["timestamp"].max())

In [ ]:
# STEP 7 — Fit the two required machine-learning algorithms

tree = DecisionTreeRegressor(
    max_depth=12,
    min_samples_leaf=8,
    random_state=42,
)

forest = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

tree.fit(X_train, y_train)
forest.fit(X_train, y_train)

pred_tree = tree.predict(X_test)
pred_forest = forest.predict(X_test)

# Simple benchmark: same hour on the previous day.
pred_baseline = test["demand_lag_24h"].to_numpy()

print("Models fitted successfully.")

In [ ]:
# STEP 8 — Evaluate with case-aligned metrics

peak_threshold = y_train.quantile(0.90)

def regression_metrics(name, y_true, y_pred, peak_threshold):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    peak_mask = y_true >= peak_threshold
    peak_mae = mean_absolute_error(y_true[peak_mask], y_pred[peak_mask])

    return {
        "model": name,
        "MAE_MW": mae,
        "RMSE_MW": rmse,
        "R2": r2,
        "Peak_MAE_MW": peak_mae,
    }

metrics = pd.DataFrame([
    regression_metrics("Previous-day baseline", y_test, pred_baseline, peak_threshold),
    regression_metrics("Decision Tree", y_test, pred_tree, peak_threshold),
    regression_metrics("Random Forest", y_test, pred_forest, peak_threshold),
])

metrics = metrics.sort_values("MAE_MW").reset_index(drop=True)

display(
    metrics.style.format({
        "MAE_MW": "{:.2f}",
        "RMSE_MW": "{:.2f}",
        "R2": "{:.4f}",
        "Peak_MAE_MW": "{:.2f}",
    })
)

print("Peak threshold derived from training data:", f"{peak_threshold:.2f} MW")

In [ ]:
# STEP 9 — Test whether the WEATHER dataset adds predictive value

NON_WEATHER_FEATURES = [c for c in FEATURE_COLS if c not in WEATHER_COLS]

forest_no_weather = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

forest_no_weather.fit(train[NON_WEATHER_FEATURES], y_train)
pred_no_weather = forest_no_weather.predict(test[NON_WEATHER_FEATURES])

rmse_with_weather = np.sqrt(mean_squared_error(y_test, pred_forest))
rmse_without_weather = np.sqrt(mean_squared_error(y_test, pred_no_weather))
mae_with_weather = mean_absolute_error(y_test, pred_forest)
mae_without_weather = mean_absolute_error(y_test, pred_no_weather)

rmse_improvement_pct = 100 * (rmse_without_weather - rmse_with_weather) / rmse_without_weather
mae_improvement_pct = 100 * (mae_without_weather - mae_with_weather) / mae_without_weather

weather_ablation = pd.DataFrame({
    "comparison": ["Random Forest without weather", "Random Forest with weather"],
    "MAE_MW": [mae_without_weather, mae_with_weather],
    "RMSE_MW": [rmse_without_weather, rmse_with_weather],
})

display(weather_ablation.style.format({"MAE_MW": "{:.2f}", "RMSE_MW": "{:.2f}"}))
print("RMSE improvement from adding weather:", f"{rmse_improvement_pct:.2f}%")
print("MAE improvement from adding weather:", f"{mae_improvement_pct:.2f}%")

In [ ]:
# STEP 10 — Feature importance and descriptive insights

feature_importance = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": forest.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(10).style.format({"importance": "{:.4f}"}))

hourly_pattern = (
    model_df.groupby("hour", as_index=False)["demand_mw"]
    .mean()
    .sort_values("demand_mw", ascending=False)
)

highest_avg_hour = int(hourly_pattern.iloc[0]["hour"])
lowest_avg_hour = int(hourly_pattern.iloc[-1]["hour"])

print("Highest average-demand hour:", highest_avg_hour)
print("Lowest average-demand hour:", lowest_avg_hour)

In [ ]:
# STEP 11 — Create figures for the appendix / executive summary

output_dir = Path("alinta_outputs")
output_dir.mkdir(exist_ok=True)

plot_df = test[["timestamp", "demand_mw"]].copy()
plot_df["Random Forest prediction"] = pred_forest
plot_df = plot_df.tail(24 * 7)

plt.figure(figsize=(11, 4.8))
plt.plot(plot_df["timestamp"], plot_df["demand_mw"], label="Actual demand")
plt.plot(plot_df["timestamp"], plot_df["Random Forest prediction"], label="Random Forest prediction")
plt.xlabel("NEM time (AEST)")
plt.ylabel("Operational demand (MW)")
plt.title("Victorian operational demand: actual vs predicted (final 7 test days)")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "actual_vs_predicted.png", dpi=220)
plt.show()

top_imp = feature_importance.head(10).sort_values("importance")

plt.figure(figsize=(8, 5))
plt.barh(top_imp["feature"], top_imp["importance"])
plt.xlabel("Feature importance")
plt.title("Random Forest: ten most influential features")
plt.tight_layout()
plt.savefig(output_dir / "feature_importance.png", dpi=220)
plt.show()

In [ ]:
# STEP 12 — Save exact report values and reusable outputs

metrics.to_csv(output_dir / "model_metrics.csv", index=False)
weather_ablation.to_csv(output_dir / "weather_ablation.csv", index=False)
feature_importance.to_csv(output_dir / "feature_importance.csv", index=False)
merged.to_csv(output_dir / "merged_hourly_data.csv", index=False)

dataset_summary = pd.DataFrame([
    {"item": "AEMO VIC1 five-minute rows in fixed study period", "value": len(vic_5min)},
    {"item": "AEMO hourly rows after aggregation", "value": len(demand_hourly)},
    {"item": "Open-Meteo hourly rows", "value": len(weather)},
    {"item": "Merged hourly rows", "value": len(merged)},
    {"item": "ML-ready rows after lag/missing-value removal", "value": len(model_df)},
    {"item": "Training rows", "value": len(train)},
    {"item": "Test rows", "value": len(test)},
])
dataset_summary.to_csv(output_dir / "dataset_summary.csv", index=False)

best_ml_row = (
    metrics[metrics["model"].isin(["Decision Tree", "Random Forest"])]
    .sort_values("MAE_MW")
    .iloc[0]
)
baseline_row = metrics.loc[metrics["model"].eq("Previous-day baseline")].iloc[0]

best_vs_baseline_mae_pct = (
    100 * (baseline_row["MAE_MW"] - best_ml_row["MAE_MW"]) / baseline_row["MAE_MW"]
)

top3 = feature_importance.head(3)["feature"].tolist()

if rmse_improvement_pct > 1:
    weather_interpretation = (
        "Weather improved Random Forest RMSE, so the two data sources provide complementary predictive information."
    )
elif rmse_improvement_pct >= -1:
    weather_interpretation = (
        "Weather changed Random Forest RMSE by very little; most predictive information came from calendar and recent-demand history."
    )
else:
    weather_interpretation = (
        "Weather worsened Random Forest RMSE in this test period; it did not add incremental predictive value beyond calendar and recent-demand history."
    )

report_text = f'''EXACT VALUES FOR PART 1.2 AND PART 1.3
=====================================

Study period: 1 September 2025 to 31 July 2026
AEMO VIC1 five-minute rows: {len(vic_5min):,}
AEMO hourly rows: {len(demand_hourly):,}
Open-Meteo hourly rows: {len(weather):,}
Merged hourly rows: {len(merged):,}
ML-ready rows: {len(model_df):,}
Training rows: {len(train):,}
Test rows: {len(test):,}

MODEL METRICS
-------------
{metrics.to_string(index=False)}

Best ML model by MAE: {best_ml_row["model"]}
Best ML MAE: {best_ml_row["MAE_MW"]:.2f} MW
Best ML RMSE: {best_ml_row["RMSE_MW"]:.2f} MW
Best ML R2: {best_ml_row["R2"]:.4f}
Best ML Peak MAE: {best_ml_row["Peak_MAE_MW"]:.2f} MW

Best ML MAE improvement over previous-day baseline: {best_vs_baseline_mae_pct:.2f}%

WEATHER ABLATION
----------------
Random Forest MAE without weather: {mae_without_weather:.2f} MW
Random Forest MAE with weather: {mae_with_weather:.2f} MW
Random Forest RMSE without weather: {rmse_without_weather:.2f} MW
Random Forest RMSE with weather: {rmse_with_weather:.2f} MW
RMSE improvement from adding weather: {rmse_improvement_pct:.2f}%
MAE improvement from adding weather: {mae_improvement_pct:.2f}%

Interpretation: {weather_interpretation}

Top three Random Forest features: {", ".join(top3)}
Highest average-demand hour: {highest_avg_hour}:00 AEST
Lowest average-demand hour: {lowest_avg_hour}:00 AEST

IMPORTANT:
- Use these exact values in the report.
- Do not describe weather as useful unless the ablation actually shows an improvement.
- Do not claim causation from feature importance.
'''

(output_dir / "REPORT_VALUES.txt").write_text(report_text, encoding="utf-8")
print(report_text)

In [ ]:
# STEP 13 — Zip the results and download them

import shutil

shutil.make_archive("alinta_outputs", "zip", output_dir)
files.download("alinta_outputs.zip")

print("Done. Keep this ZIP. It contains the exact values, tables and figures for your report.")